<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.3-controlled-generation/notebooks/exercises-4_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.3: Controlled Generation: ControlNet, LoRA & IP-Adapter

*Module 4 · Image, Vision & Multimodal AI*

> Text prompts give rough control. ControlNet adds spatial guides (edges, poses). LoRA trains custom styles in 20 minutes. IP-Adapter uses images as prompts. Together: precision engineering for AI art.

`ControlNet` · `LoRA Training` · `IP-Adapter` · `Vertex AI` · `75 min`

---

## About this notebook

This notebook contains the **6 runnable code examples** from the published lesson `lesson-4.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — ControlNet — Guide with Structure — `01_controlnet.py`
2. Step 2 — LoRA Training — Custom Styles in 20 Minutes — `02_lora_train.py`
3. Step 3 — IP-Adapter — Image as Prompt — `03_ip_adapter.py`
4. Step 4 — Inpainting — Mask-Based Editing — `inpainting.py`
5. Step 5 — Multi-ControlNet & Union Models — Combine Control Signals — `multi_controlnet.py`
6. Step 8 — Regional Prompting — Different Prompts for Different Regions — `regional_prompting.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q torch pillow accelerate


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · ControlNet — Guide with Structure

Extract edges/pose/depth from a source, generate new content following that structure.

**`01_controlnet.py`** · _Block 1 of 6_

CONTROLNET — Spatial control over Stable Diffusion — pip install diffusers controlnet-aux


In [ ]:
# CONTROLNET — Spatial control over Stable Diffusion
# pip install diffusers controlnet-aux
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
from controlnet_aux import CannyDetector

# Step 1: Extract edges
canny = CannyDetector()
source = load_image("https://example.com/building.png")
edges = canny(source, low_threshold=100, high_threshold=200)

# Step 2: Load ControlNet + SDXL
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet, torch_dtype=torch.float16).to("cuda")

# Step 3: Generate conditioned on edges
image = pipe(
    prompt="A futuristic neon cityscape, cyberpunk, detailed",
    negative_prompt="blurry, low quality",
    image=edges,
    controlnet_conditioning_scale=0.7,  # 0=ignore, 1=rigid
    num_inference_steps=30,
).images[0]
image.save("controlnet_output.png")
print("Output follows SHAPE of original with new content!")


> **What just happened?** Canny edges extracted structure from the source. ControlNet forced generation to follow those edges. conditioning_scale=0.7 = creative but guided. Condition types: Canny (outlines), OpenPose (body pose), depth map (3D), segmentation (regions), scribble (hand-drawn).


### Step 2 · LoRA Training — Custom Styles in 20 Minutes

Fine-tune 0.1% of parameters. 15-30 images. Result: ~30 MB weights file.

**`02_lora_train.py`** · _Block 2 of 6_

LoRA TRAINING + INFERENCE — Training: accelerate launch train_dreambooth_lora_sdxl.py \


In [ ]:
# LoRA TRAINING + INFERENCE
# Training: accelerate launch train_dreambooth_lora_sdxl.py \
#   --pretrained_model="stabilityai/stable-diffusion-xl-base-1.0" \
#   --instance_data_dir="./train_data" \
#   --instance_prompt="a photo in sks style" \
#   --output_dir="./lora_weights" \
#   --resolution=1024 --train_batch_size=1 \
#   --learning_rate=1e-4 --max_train_steps=1000 --rank=32

# Inference: load and use the tiny LoRA weights
import torch
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16).to("cuda")

pipe.load_lora_weights("./lora_weights")  # ~30 MB!

image = pipe("A sunset over Hyderabad in sks style",
    num_inference_steps=30).images[0]
image.save("lora_output.png")

print("Base: 6.5 GB | LoRA: 30 MB | Trigger: 'sks style'")
print("Stack multiple LoRAs. Switch styles. Combine them.")


### Step 3 · IP-Adapter — Image as Prompt

Show a reference image. The model transfers its style, color palette, mood.

**`03_ip_adapter.py`** · _Block 3 of 6_

IP-ADAPTER — Use an image as a visual prompt


In [ ]:
# IP-ADAPTER — Use an image as a visual prompt
import torch
from diffusers import StableDiffusionXLPipeline
from diffusers.utils import load_image

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16).to("cuda")

pipe.load_ip_adapter("h94/IP-Adapter",
    subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")

ref = load_image("reference_style.jpg")
pipe.set_ip_adapter_scale(0.6)  # 0=ignore image, 1=copy exactly

image = pipe(
    prompt="A majestic palace in the mountains, golden hour",
    ip_adapter_image=ref,
    num_inference_steps=30).images[0]
image.save("ip_adapter_output.png")

print("Text: content (palace). Image: style (reference colors/texture)")
print("Zero training needed — works with any reference!")


> **What just happened?** IP-Adapter combined text prompt (content) with image prompt (style). Scale 0.6 = balanced influence. Unlike LoRA: zero training. Just pass any reference image. The model transfers color palette, texture, and mood.


### Step 4 · Inpainting — Mask-Based Editing

Paint a mask. Regenerate only that region. Object removal, background replacement, product placement.

**`inpainting.py`** · _Block 4 of 6_


In [ ]:
from diffusers import AutoPipelineForInpainting
import torch
from PIL import Image

# Load inpainting pipeline (auto-detects correct class)
pipe = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    torch_dtype=torch.float16, variant="fp16"
).to("cuda")

# Load image + mask (white=regenerate, black=keep)
image = Image.open("room.jpg").resize((1024, 1024))
mask = Image.open("mask.png").resize((1024, 1024))

result = pipe(
    prompt="A modern sofa with cushions, warm lighting",
    image=image,
    mask_image=mask,
    strength=0.99,          # 0.99, not 1.0 (avoids VAE noise artifact)
    guidance_scale=8.0,
    num_inference_steps=25,
    padding_mask_crop=32,    # Crop around mask for better small-area quality
).images[0]
result.save("room_new_sofa.png")

# OUTPAINTING: extend image beyond boundaries
# 1. Pad the original image with blank pixels
# 2. Create mask over the padded area
# 3. Run inpainting → seamless extension
# Best: FLUX.1 Fill [dev] — native inpaint/outpaint, state-of-the-art
# from diffusers import FluxFillPipeline
# pipe = FluxFillPipeline.from_pretrained("black-forest-labs/FLUX.1-Fill-dev")


> **What just happened?** Inpainting = mask-based editing. White pixels in the mask are regenerated, black pixels are preserved. strength=0.99 (not 1.0!) starts from near-pure noise for maximum creativity while avoiding a VAE autoencoding artifact. padding_mask_crop=32 crops around the mask, upscales for inpainting, then composites back — dramatically improves small-area quality. FLUX.1 Fill is the 2026 state-of-the-art for inpainting/outpainting — no strength parameter needed, native 12B transformer quality. Use cases: object removal, background replacement, product placement, furniture visualization.


### Step 5 · Multi-ControlNet & Union Models — Combine Control Signals

Canny + depth + pose simultaneously. Or one Union model that handles all types.

**`multi_controlnet.py`** · _Block 5 of 6_


In [ ]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline

# Load multiple ControlNets
controlnets = [
    ControlNetModel.from_pretrained("diffusers/controlnet-depth-sdxl-1.0-small"),
    ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0"),
]
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnets, torch_dtype=torch.float16
).to("cuda")

result = pipe(
    prompt="Modern glass office building, sunset sky",
    image=[depth_map, canny_edges],         # One image per ControlNet
    controlnet_conditioning_scale=[0.8, 0.5], # Weight per model
    control_guidance_start=[0.0, 0.0],       # When to start (0-1)
    control_guidance_end=[0.8, 0.5],         # When to stop (0-1)
).images[0]

# ═══════ UNION MODELS: One model handles ALL types ═══════
# SDXL: xinsir/controlnet-union-sdxl-1.0 (~2.5GB, 12+ control types)
# Flux: Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0 (~4GB, 5 modes)
# → No need to load separate models for each control type!


> **What just happened?** Multi-ControlNet combines multiple control signals by passing a list of models, images, and scales. ~70% of production requests use 2+ ControlNets. The control_guidance_end parameter is crucial: setting it to 0.5-0.8 lets ControlNet guide structure in early steps, then the model refines details freely. Union models (ControlNet++) handle 12+ control types in a single ~2.5GB model — eliminating the need to load separate models. For Flux: Shakker-Labs Union Pro 2.0 supports canny, depth, pose, softedge with per-mode optimal parameters.


### Step 8 · Regional Prompting — Different Prompts for Different Regions

Left side: "woman in red saree." Right side: "man in blue kurta." No color leaking.

**`regional_prompting.py`** · _Block 6 of 6_

═══════ REGIONAL PROMPTING ═══════ — Problem: "man with black hair and woman with blonde hair"


In [ ]:
# ═══════ REGIONAL PROMPTING ═══════
# Problem: "man with black hair and woman with blonde hair"
#   → Model swaps colors! (attribute binding failure)
# Solution: assign different prompts to different image regions

# In ComfyUI: Attention Couple nodes (A8R8)
# Each region gets: conditioning + binary mask + weight
# AttentionCoupleRegion → AttentionCouple → patches the model

# In A1111: Regional Prompter extension
# Syntax: "left prompt BREAK right prompt"
# Modes: Attention (fast, 8×8 resolution)
#        Latent (slower, supports per-region LoRAs)

# In diffusers: ConditioningSetArea
# Or use Compel library for prompt weighting: (word:1.5)
# Weight range: 0.5-1.5 (beyond 1.5 = artifacts)

# Attend-and-Excite (SIGGRAPH 2023):
# Prevents "catastrophic neglect" — when model forgets subjects
# Boosts cross-attention for neglected tokens during denoising
# from diffusers import StableDiffusionAttendAndExcitePipeline


> **What just happened?** Regional prompting assigns different text prompts to different image areas — solving the #1 multi-subject failure (attribute confusion). In ComfyUI: Attention Couple nodes with binary masks per region. Prompt weighting syntax (word:1.5) increases emphasis (0.5-1.5 range). Attend-and-Excite prevents subject neglect by boosting cross-attention for specific tokens. Best practice for multi-subject: plan layout → create masks → Attention Couple with per-region prompts → OpenPose ControlNet for positioning.


---

## Wrap-up

You've walked through all 6 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
